In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from helpers import (
    load_h5_file,
    load_envue_mat_file,
    remove_background,
    calculate_probability_map,
    segmented_multi_track_viterbi,
    link_segmented_tracks_with_viterbi,
    ParticleTrack
)
from pathlib import Path


## Parameters
Exported from h5_viewer with file: `100nm_4.h5`


In [ ]:
# File and loading parameters
file_path = r'/home/nttadm/Repos/nsm-control/data/100nm_4.h5'
load_func = load_h5_file
binning_x = 2
binned_time_ms = 2

# Background removal parameters
average_samples_x = 20
average_samples_t = 50

# Spatial cropping
crop_first_pixels = 0
crop_last_pixels = 0

# Probability map parameters
blurring_sigma = 2
prior_particle_probability = 0.3

# Track finding parameters
segment_length = 100
overlap_length = 50
expected_diffusion_sigma = 1.0
velocity = 0.8
num_passes = 3

# Track linking parameters
sigma_gap = 2.0
max_average_gap = 5.0
min_path_length = 10
edge_region_percent = 5.0


## Load Data


In [ ]:
kymo, frame_rate = load_func(file_path, binning_x=binning_x, binned_time_ms=binned_time_ms)
print(f'Loaded data shape: {kymo.shape}, frame rate: {frame_rate:.2f} fps')

# Apply spatial cropping
if crop_first_pixels > 0 or crop_last_pixels > 0:
    num_pixels = kymo.shape[1]
    crop_end = num_pixels - crop_last_pixels if crop_last_pixels > 0 else num_pixels
    kymo = kymo[:, crop_first_pixels:crop_end]
    print(f'Cropped to shape: {kymo.shape}')


## Background Removal


In [ ]:
kymo_cp = cp.asarray(kymo, dtype=cp.float32)
bg_removed = remove_background(kymo_cp, average_samples_x, average_samples_t)
print(f'Background removed. Shape: {bg_removed.shape}')


## Probability Map


In [ ]:
prob_map = calculate_probability_map(bg_removed, blurring_sigma, prior_particle_probability)
print(f'Probability map calculated. Shape: {prob_map.shape}')


## Visualize Kymograph


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(cp.asnumpy(bg_removed[:1000]).T, aspect='auto', cmap='gray')
axes[0].set_title('Background Removed (first 1000 frames)')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Pixel')
axes[1].imshow(cp.asnumpy(prob_map[:1000]).T, aspect='auto', cmap='gray')
axes[1].set_title('Probability Map (first 1000 frames)')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Pixel')
plt.tight_layout()
plt.show()


## Find Tracks in Segments


In [ ]:
segmented_tracks, segmented_probabilities, all_log_probs = segmented_multi_track_viterbi(
    prob_map,
    segment_length=segment_length,
    overlap_length=overlap_length,
    sigma_transition=expected_diffusion_sigma,
    velocity=velocity,
    num_passes=num_passes
)

# Filter edge tracks
if edge_region_percent > 0:
    num_pixels = prob_map.shape[1]
    edge_width = num_pixels * (edge_region_percent / 100.0)
    lower_bound = edge_width
    upper_bound = num_pixels - edge_width
    filtered_seg_tracks = []
    filtered_seg_probs = []
    for seg_tracks, seg_probs in zip(segmented_tracks, segmented_probabilities):
        kept_t, kept_p = [], []
        for track, prob in zip(seg_tracks, seg_probs):
            if not (np.max(track) <= lower_bound or np.min(track) >= upper_bound):
                kept_t.append(track)
                kept_p.append(prob)
        filtered_seg_tracks.append(kept_t)
        filtered_seg_probs.append(kept_p)
    segmented_tracks = filtered_seg_tracks
    segmented_probabilities = filtered_seg_probs

total_tracks = sum(len(t) for t in segmented_tracks)
print(f'Found {total_tracks} segment tracks in {len(segmented_tracks)} segments')


## Link Tracks Across Segments


In [ ]:
linked_tracks = link_segmented_tracks_with_viterbi(
    segmented_tracks,
    segmented_probabilities,
    segment_length,
    overlap_length,
    sigma_gap,
    max_average_gap,
    min_path_length,
    prob_map=prob_map
)

# Trim tracks
trimmed_tracks = [
    track.trim_track_by_probability(threshold_factor=1, window_width=segment_length)
    for track in linked_tracks
]
trimmed_tracks = [t for t in trimmed_tracks if t is not None]
print(f'Linked: {len(linked_tracks)} tracks, Trimmed: {len(trimmed_tracks)} tracks')


## Visualize Tracks on Kymograph


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
kymo_display = cp.asnumpy(prob_map) if hasattr(prob_map, 'get') else prob_map
ax.imshow(kymo_display.T, aspect='auto', cmap='gray', origin='lower')

for track in trimmed_tracks:
    t_start = track.start_time
    positions = track.stitched_track
    frames = np.arange(t_start, t_start + len(positions))
    ax.plot(frames, positions, linewidth=0.8, alpha=0.7)

ax.set_xlabel('Frame')
ax.set_ylabel('Pixel Position')
ax.set_title(f'Trimmed Tracks ({len(trimmed_tracks)} tracks)')
plt.tight_layout()
plt.show()


## Track Statistics


In [ ]:
diffusion_coeffs = []
contrasts = []
track_lengths = []
mean_velocities = []

for track in trimmed_tracks:
    diff_result = track.calculate_diffusion(
        framerate=frame_rate, pixel_size=1.0, remove_drift=True
    )
    if diff_result is not None:
        contrast_result = track.calculate_image_contrast_along_track(
            bg_removed, gaussian_blur_sigma=2
        )
        if contrast_result is not None:
            mean_contrast, _ = contrast_result
            if hasattr(mean_contrast, 'get'):
                mean_contrast = mean_contrast.get()
            diffusion_coeffs.append(diff_result['D'])
            contrasts.append(float(mean_contrast))
            track_lengths.append(len(track))
            positions = track.stitched_track
            time_s = np.arange(len(positions)) / frame_rate
            vel = np.polyfit(time_s, positions, 1)[0]
            mean_velocities.append(vel)

diffusion_coeffs = np.array(diffusion_coeffs)
contrasts = np.array(contrasts)
track_lengths = np.array(track_lengths)
mean_velocities = np.array(mean_velocities)
print(f'Computed statistics for {len(diffusion_coeffs)} tracks')


## Histograms


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(diffusion_coeffs, bins=30, edgecolor='black')
axes[0, 0].set_xlabel('Diffusion Coefficient (pixels\u00b2/s)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Diffusion Coefficients')

axes[0, 1].hist(contrasts, bins=30, edgecolor='black')
axes[0, 1].set_xlabel('Mean Contrast (a.u.)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Mean Contrast')

axes[1, 0].hist(track_lengths, bins=30, edgecolor='black')
axes[1, 0].set_xlabel('Track Length (frames)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Track Lengths')

axes[1, 1].hist(mean_velocities, bins=30, edgecolor='black')
axes[1, 1].set_xlabel('Mean Velocity (pixels/s)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Mean Velocities')

plt.tight_layout()
plt.show()


## Diffusion vs Contrast Scatter


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(contrasts, diffusion_coeffs, c=track_lengths,
                     cmap='viridis', alpha=0.6, edgecolors='black', linewidth=0.5)
ax.set_xlabel('Mean Contrast (a.u.)')
ax.set_ylabel('Diffusion Coefficient (pixels\u00b2/s)')
ax.set_title(f'Diffusion vs Contrast (N={len(diffusion_coeffs)})')
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.colorbar(scatter, label='Track Length (frames)')
plt.tight_layout()
plt.show()


## Further Analysis
The `trimmed_tracks` list contains `ParticleTrack` objects with:
- `track.stitched_track` - positions array
- `track.start_time` - start frame
- `track.time_indices` - frame indices
- `track.calculate_diffusion(framerate, pixel_size, remove_drift)` - MSD-based diffusion
- `track.calculate_image_contrast_along_track(image, gaussian_blur_sigma)` - contrast

Filter, group, or further process the tracks below.


In [ ]:
# Example: filter tracks by velocity range
# velocity_min, velocity_max = -100, -20
# mask = (mean_velocities > velocity_min) & (mean_velocities < velocity_max)
# filtered_tracks = [t for t, m in zip(trimmed_tracks, mask) if m]
# print(f'Filtered to {len(filtered_tracks)} tracks')
